# Hands-on: Fine-Tuning com LoRA (BERTimbau + PEFT)

Exemplo pequeno para acompanhar o bloco 5 da aula (*Reparametrização: LoRA e QLoRA*).

**O que vamos fazer:**
1. Carregar um BERT pré-treinado em português (BERTimbau).
2. Ver quantos parâmetros um fine-tuning completo atualizaria.
3. Aplicar LoRA e comparar o número de parâmetros treináveis.
4. Treinar um classificador de texto curto (3 classes) em poucos minutos.
5. Testar o modelo com frases novas.

**Dataset:** um conjunto pequeno e **sintético** de frases no estilo de anotações clínicas curtas, criado só para fins didáticos (não são dados reais de pacientes), classificando cada frase em um de três grupos temáticos: respiratório, circulatório, transtornos mentais.

> Execução no Colab: `Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4)` antes de rodar.


In [ ]:
!pip install -q transformers datasets peft accelerate scikit-learn


In [ ]:
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

torch.manual_seed(42)


## 1. Dataset (sintético, apenas para fins didáticos)

25 frases por classe. Nada aqui é dado real de paciente — são frases curtas escritas para simular o estilo de uma anotação clínica.


In [ ]:
respiratorio = [
    "Paciente relata tosse seca persistente há cinco dias.",
    "Refere falta de ar ao subir escadas.",
    "Ausculta pulmonar com sibilos difusos bilaterais.",
    "Quadro compatível com bronquite aguda.",
    "Paciente apresenta dispneia progressiva há uma semana.",
    "Radiografia de tórax sugestiva de pneumonia em base direita.",
    "Tosse produtiva com expectoração amarelada.",
    "Paciente asmático em crise, uso de broncodilatador.",
    "Relata chiado no peito à noite.",
    "Diagnóstico de rinossinusite com obstrução nasal.",
    "Saturação de oxigênio reduzida ao esforço.",
    "Paciente com histórico de DPOC apresenta piora da dispneia.",
    "Tosse seca associada a dor torácica ventilatório-dependente.",
    "Exame confirma derrame pleural à direita.",
    "Paciente relata coriza e espirros frequentes.",
    "Quadro gripal com congestão nasal e tosse.",
    "Ausculta revela estertores crepitantes em base pulmonar.",
    "Paciente com faringite e rouquidão há três dias.",
    "Episódio de broncoespasmo após exposição a poeira.",
    "Paciente com apneia do sono relata sonolência diurna.",
    "Diagnóstico de sinusite crônica com dor facial.",
    "Falta de ar súbita associada a dor pleurítica.",
    "Tosse crônica em paciente tabagista há vinte anos.",
    "Paciente com laringite apresenta rouquidão intensa.",
    "Exame de imagem mostra nódulo pulmonar a investigar.",
]

circulatorio = [
    "Paciente hipertenso relata cefaleia occipital pela manhã.",
    "Queixa de dor torácica em aperto ao esforço.",
    "Eletrocardiograma mostra arritmia sinusal.",
    "Paciente apresenta edema bilateral de membros inferiores.",
    "Pressão arterial aferida em 160 por 100 mmHg.",
    "Relata palpitações frequentes nas últimas semanas.",
    "Paciente com histórico de infarto do miocárdio prévio.",
    "Ausculta cardíaca com sopro sistólico audível.",
    "Queixa de tontura ao levantar associada a hipotensão postural.",
    "Paciente varicoso relata dor e peso nas pernas.",
    "Exame revela insuficiência cardíaca congestiva descompensada.",
    "Paciente com fibrilação atrial em uso de anticoagulante.",
    "Dor torácica irradiando para o braço esquerdo.",
    "Paciente diabético com risco cardiovascular elevado.",
    "Relata dispneia aos esforços associada a cansaço.",
    "Extremidades frias com pulsos periféricos diminuídos.",
    "Paciente com angina estável em tratamento medicamentoso.",
    "Exame mostra hipertrofia ventricular esquerda.",
    "Queixa de dor em panturrilha sugestiva de trombose venosa.",
    "Paciente relata episódios de taquicardia após esforço físico.",
    "Ecocardiograma evidencia disfunção diastólica leve.",
    "Paciente com colesterol elevado e histórico familiar de infarto.",
    "Relata desmaio precedido de sensação de aperto no peito.",
    "Paciente com aneurisma de aorta abdominal em acompanhamento.",
    "Edema facial associado a aumento da pressão arterial.",
]

mental = [
    "Paciente relata humor deprimido e desânimo há duas semanas.",
    "Queixa de insônia inicial há vários dias.",
    "Refere crises de ansiedade com falta de ar e taquicardia.",
    "Paciente relata perda de interesse em atividades habituais.",
    "Sintomas compatíveis com transtorno de pânico.",
    "Paciente apresenta irritabilidade e dificuldade de concentração.",
    "Relata pensamentos acelerados e dificuldade para dormir.",
    "Paciente em acompanhamento por transtorno de ansiedade generalizada.",
    "Queixa de fadiga mental e desmotivação constante.",
    "Relata episódios de choro fácil sem motivo aparente.",
    "Paciente apresenta sintomas de estresse pós-traumático.",
    "Refere dificuldade de concentração no trabalho há semanas.",
    "Paciente relata isolamento social crescente.",
    "Queixa de preocupação excessiva com o futuro.",
    "Paciente com histórico de transtorno bipolar em remissão.",
    "Relata sensação constante de nervosismo e tensão muscular.",
    "Paciente apresenta alterações de apetite associadas ao humor.",
    "Refere pensamentos ruminantes recorrentes à noite.",
    "Paciente relata dificuldade em lidar com situações de estresse.",
    "Queixa de sensação de vazio e falta de motivação.",
    "Paciente apresenta sintomas obsessivo-compulsivos leves.",
    "Relata insônia terminal com despertar precoce.",
    "Paciente em crise de ansiedade antes de eventos sociais.",
    "Queixa de memória prejudicada associada a quadro ansioso.",
    "Paciente relata sensação de desesperança nos últimos dias.",
]

label_names = ["respiratorio", "circulatorio", "transtornos_mentais"]
textos = respiratorio + circulatorio + mental
rotulos = [0] * len(respiratorio) + [1] * len(circulatorio) + [2] * len(mental)

print(f"Total de exemplos: {len(textos)}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    textos, rotulos, test_size=0.2, random_state=42, stratify=rotulos
)

train_ds = Dataset.from_dict({"text": X_train, "label": y_train})
test_ds = Dataset.from_dict({"text": X_test, "label": y_test})

print(f"Treino: {len(train_ds)} exemplos | Teste: {len(test_ds)} exemplos")


## 2. Tokenizador e modelo base

Usamos o **BERTimbau base** (`neuralmind/bert-base-portuguese-cased`), um BERT pré-treinado em português — o mesmo tipo de modelo discutido no bloco 1 da aula.


In [ ]:
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=48)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names)
)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Parâmetros totais do modelo: {total_params:,}")
print("Num fine-tuning completo, TODOS esses parâmetros seriam atualizados.")


## 3. Aplicando LoRA

Relembrando a fórmula do bloco 5: em vez de atualizar $W_0$ inteira, LoRA aprende duas matrizes pequenas $A$ e $B$ tal que $W = W_0 + BA$, com $W_0$ congelada.

Aqui aplicamos LoRA nas camadas `query` e `value` da atenção — os mesmos módulos usados no artigo original (Hu et al., 2021).


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, #rank das matrizes A e B
    lora_alpha=16, # fator de escala (alpha / r)
    lora_dropout=0.1,
    target_modules=["query", "value"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()


## 4. Treinamento

Poucas épocas, dataset pequeno de propósito — o objetivo aqui é ilustrar o mecanismo, não bater estado da arte.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./lora-bertimbau-demo",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
results = trainer.evaluate()
print(results)

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
print(classification_report(y_test, y_pred, target_names=label_names))


## 5. Testando com frases novas


In [ ]:
frases_novas = [
    "Paciente relata dor no peito e falta de ar aos esforços.",
    "Refere insônia e tristeza constante nas últimas semanas.",
    "Tosse persistente com chiado no peito há alguns dias.",
]

lora_model.eval()
inputs = tokenizer(frases_novas, padding=True, truncation=True, max_length=48, return_tensors="pt")
inputs = {k: v.to(lora_model.device) for k, v in inputs.items()}

with torch.no_grad():
    logits = lora_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)

for frase, pred in zip(frases_novas, pred_ids):
    print(f"{frase}\n  -> classe prevista: {label_names[pred.item()]}\n")


## 6. Ponto de discussão para a turma

- Compare o número impresso em `print_trainable_parameters()` com o total de parâmetros do modelo (célula da seção 2). Que percentual isso representa?
- O adaptador LoRA (`lora_model.save_pretrained(...)`) ocupa poucos MB em disco — bem diferente de salvar o BERT inteiro fine-tuned. Vale rodar `lora_model.save_pretrained("./adapter")` e checar o tamanho da pasta com `!du -sh ./adapter`.
- **Exercício:** repita o treino mudando `r` (ex.: 2, 4, 32) e observe o impacto no número de parâmetros treináveis e na acurácia final. É um jeito rápido de sentir na prática o trade-off discutido no bloco 5 da aula.
- Com pouco mais de tempo, dá para comparar este resultado com um fine-tuning completo (sem LoRA) do mesmo modelo, para a turma ver o ganho de eficiência lado a lado.
